# 04 - Final Recall Test (3-slice v2)

Évaluation finale avec ResNet-50 ImageNet + crop_margin=30 + 3-slice. Écrit `data/results/dice_final_report_resnet50_wide_crop.csv`.

In [ ]:
#@title Colab setup
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/moradBMH/INF8225_Projet.git"
GIT_REF = "main"
PROJECT_DIR = Path("/content/INF8225_Projet")
DRIVE_FOLDER = None
INSTALL_DEPS = True
REINSTALL_DEPS = False

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from notebooks.colab_setup import setup
setup(drive_folder=DRIVE_FOLDER, install=INSTALL_DEPS, reinstall=REINSTALL_DEPS)
print("cwd:", Path.cwd())

In [ ]:
#@title Verify 3-slice v2 artifacts and MedSAM
from pathlib import Path
from msd_recall_strategy import get_resnet_checkpoint_dir

checkpoint_dir = get_resnet_checkpoint_dir()
required = [
    Path("data/MSD_pancreas/test.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("work_dirs/tumor_config_v3/tumor_config_v3.py"),
    Path("MedSAM/work_dir/MedSAM/medsam_vit_b.pth"),
    Path("optimal_threshold_resnet50_wide_crop.txt"),
]
required += [checkpoint_dir / f"resnet50_wide_crop_fold_{i}.pth" for i in range(1, 6)]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}. Lance d'abord les notebooks 01, 02, 03 v2."

In [ ]:
#@title Run pipeline step (3-slice v2 test)
from collections import deque
import subprocess
import sys

cmd = [sys.executable, "-u", "-m", "experiments.msd.resnet50_wide_crop.evaluate"]
print("Running:", " ".join(cmd))

last_lines = deque(maxlen=160)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
    last_lines.append(line)

return_code = proc.wait()
if return_code != 0:
    print("\n" + "=" * 80)
    print(f"test_v2 failed with exit code {return_code}. Last captured lines:")
    print("=" * 80)
    print("".join(last_lines))
    raise RuntimeError(f"test_v2 failed with exit code {return_code}")

In [ ]:
#@title Inspect final 3-slice v2 report
from pathlib import Path
import pandas as pd
report = Path("data/results/dice_final_report_resnet50_wide_crop.csv")
print(("OK     " if report.exists() else "MISSING"), report)
assert report.exists(), "Final 3-slice v2 report CSV missing"
df = pd.read_csv(report)
display(df.head())
print("rows:", len(df))
print("mean dice:", df["final_dice"].mean())